# Sequence Negative Control Experiments

Ce notebook reprend le script `sequence_negative_control_experiments.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Controle negatif pour verifier que le modele sequence live n'apprend pas un signal artificiel.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Parent-video label-shuffle negative control for sequence danger models.
- Commande de reproduction referencee : exact negative control.
- Artefacts controles : Exact-prevalence sequence label-shuffle negative control exists. (`runs/exp_025_sequence_negative_control_exact/metrics/negative_control_summary.csv`).
- Run par defaut : `runs/exp_024_sequence_negative_control`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "sequence_negative_control_experiments.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import json
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from torch.utils.data import DataLoader

from ml_pipeline import HORIZONS, ROOT, safe_auc, write_json
from sequence_experiments import (
    SequenceDataset,
    append_report,
    make_run_dir,
    predict_model,
    train_one_model,
)
from sequence_feature_ablation_experiments import build_feature_groups, load_sequence


## Fonction `make_parent_sequence_shuffle`

Cette cellule definit `make_parent_sequence_shuffle`. Elle prepare une partie du script.

In [ ]:
def make_parent_sequence_shuffle(y, meta, seed):
    """Shuffle label sequences between parent videos inside each split.

    This breaks feature-label alignment while preserving split membership, class
    prevalence approximately, and the fact that positives appear in contiguous
    temporal regions rather than as isolated random rows.
    """
    rng = np.random.default_rng(seed)
    shuffled = np.zeros_like(y)
    manifest = []
    for split, split_df in meta.groupby("split", sort=True):
        videos = sorted(split_df["video_id"].unique().tolist())
        permuted = videos.copy()
        if len(permuted) > 1:
            for _ in range(20):
                rng.shuffle(permuted)
                if any(a != b for a, b in zip(videos, permuted)):
                    break
        source_by_target = dict(zip(videos, permuted))
        for target_video in videos:
            source_video = source_by_target[target_video]
            target_idx = meta.index[meta["video_id"] == target_video].to_numpy()
            source_idx = meta.index[meta["video_id"] == source_video].to_numpy()
            target_idx = target_idx[np.argsort(meta.loc[target_idx, "time_s"].to_numpy(dtype=float))]
            source_idx = source_idx[np.argsort(meta.loc[source_idx, "time_s"].to_numpy(dtype=float))]
            if len(source_idx) == 0 or len(target_idx) == 0:
                continue
            take = np.linspace(0, len(source_idx) - 1, len(target_idx)).round().astype(int)
            shuffled[target_idx] = y[source_idx[take]]
            manifest.append(
                {
                    "split": split,
                    "target_video_id": target_video,
                    "source_video_id": source_video,
                    "target_windows": int(len(target_idx)),
                    "source_windows": int(len(source_idx)),
                    "target_positive_1s_original": int(y[target_idx, HORIZONS.index(1.0)].sum()),
                    "assigned_positive_1s_control": int(shuffled[target_idx, HORIZONS.index(1.0)].sum()),
                }
            )
    return shuffled.astype(np.float32), pd.DataFrame(manifest)


## Fonction `make_exact_window_shuffle`

Cette cellule definit `make_exact_window_shuffle`. Elle prepare une partie du script.

In [ ]:
def make_exact_window_shuffle(y, meta, seed):
    """Shuffle window labels inside each split and horizon, preserving exact counts."""
    rng = np.random.default_rng(seed)
    shuffled = np.zeros_like(y)
    manifest = []
    for split, split_df in meta.groupby("split", sort=True):
        split_idx = split_df.index.to_numpy()
        for h_idx, horizon in enumerate(HORIZONS):
            labels = y[split_idx, h_idx].copy()
            rng.shuffle(labels)
            shuffled[split_idx, h_idx] = labels
            manifest.append(
                {
                    "split": split,
                    "horizon_s": float(horizon),
                    "windows": int(len(split_idx)),
                    "positive_original": int(y[split_idx, h_idx].sum()),
                    "positive_control": int(shuffled[split_idx, h_idx].sum()),
                }
            )
    return shuffled.astype(np.float32), pd.DataFrame(manifest)


## Fonction `control_meta`

Cette cellule definit `control_meta`. Elle prepare une partie du script.

In [ ]:
def control_meta(meta, y_control):
    out = meta.copy()
    for h_idx, horizon in enumerate(HORIZONS):
        out[f"danger_within_{horizon:.1f}s"] = y_control[:, h_idx].astype(int)
    out["is_danger_clip"] = (
        out.groupby("video_id")["danger_within_1.0s"].transform("max").astype(int)
    )
    out["target_time_s"] = ""
    out["time_to_target_s"] = ""
    return out


## Fonction `best_f1_threshold`

Cette cellule definit `best_f1_threshold`. Elle prepare une partie du script.

In [ ]:
def best_f1_threshold(y_true, y_score):
    best = {"threshold": 0.5, "f1": 0.0}
    for threshold in np.arange(0.05, 1.0, 0.05):
        pred = (y_score >= threshold).astype(int)
        score = float(f1_score(y_true, pred, zero_division=0))
        if score > best["f1"]:
            best = {"threshold": float(round(threshold, 2)), "f1": score}
    return best


## Fonction `evaluate_probs`

Cette cellule definit `evaluate_probs`. Elle prepare une partie du script.

In [ ]:
def evaluate_probs(name, probs, labels_by_source, meta, feature_set, spec, shuffle_seed):
    rows = []
    for label_source, labels in labels_by_source.items():
        for split in ["train", "val", "test"]:
            mask = meta["split"].to_numpy() == split
            for h_idx, horizon in enumerate(HORIZONS):
                y_true = labels[mask, h_idx].astype(int)
                y_score = probs[mask, h_idx]
                best = best_f1_threshold(y_true, y_score)
                rows.append(
                    {
                        "model": name,
                        "feature_set": feature_set,
                        "spec": spec,
                        "shuffle_seed": int(shuffle_seed),
                        "label_source": label_source,
                        "split": split,
                        "horizon_s": float(horizon),
                        "n": int(mask.sum()),
                        "positive": int(y_true.sum()),
                        "prevalence": float(y_true.mean()) if len(y_true) else 0.0,
                        "average_precision": safe_auc(average_precision_score, y_true, y_score),
                        "roc_auc": safe_auc(roc_auc_score, y_true, y_score),
                        "best_f1": best["f1"],
                        "best_f1_threshold": best["threshold"],
                    }
                )
    return rows


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(metrics, run_dir):
    h1 = metrics[metrics["horizon_s"].astype(float).eq(1.0)].copy()
    summary_rows = []
    for (label_source, split, feature_set, spec), group in h1.groupby(["label_source", "split", "feature_set", "spec"]):
        summary_rows.append(
            {
                "label_source": label_source,
                "split": split,
                "feature_set": feature_set,
                "spec": spec,
                "runs": int(group["shuffle_seed"].nunique()),
                "prevalence_mean": float(group["prevalence"].mean()),
                "ap_mean": float(group["average_precision"].mean()),
                "ap_std": float(group["average_precision"].std(ddof=0)),
                "roc_auc_mean": float(group["roc_auc"].mean()),
                "roc_auc_std": float(group["roc_auc"].std(ddof=0)),
                "best_f1_mean": float(group["best_f1"].mean()),
            }
        )
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(run_dir / "metrics" / "negative_control_summary.csv", index=False)

    control_test = summary[(summary["label_source"] == "control_shuffled") & (summary["split"] == "test")].sort_values("ap_mean", ascending=False)
    real_test = summary[(summary["label_source"] == "real_original") & (summary["split"] == "test")].sort_values("ap_mean", ascending=False)
    mode = str(metrics["shuffle_mode"].iloc[0]) if "shuffle_mode" in metrics and len(metrics) else "unknown"
    lines = ["# Sequence Label-Shuffle Negative Control", ""]
    if mode == "parent_sequence":
        lines.append("Labels are shuffled as whole temporal label sequences between parent videos inside each split. This preserves split membership and rough temporal label structure, but can distort prevalence when source/target clip lengths differ.")
    elif mode == "window_exact":
        lines.append("Window labels are shuffled inside each split and horizon. This preserves the exact positive count for every split/horizon but breaks temporal label structure and feature-label alignment.")
    else:
        lines.append("Labels are deliberately shuffled to break the real feature-label relationship.")
    lines.append("")
    lines.append("## Test Metrics Against Shuffled Control Labels")
    lines.append("")
    lines.append("| feature set | spec | runs | prevalence | AP mean | AP std | ROC AUC mean | F1 mean |")
    lines.append("|---|---|---:|---:|---:|---:|---:|---:|")
    for _, row in control_test.iterrows():
        lines.append(
            f"| {row['feature_set']} | {row['spec']} | {int(row['runs'])} | {row['prevalence_mean']:.3f} | {row['ap_mean']:.3f} | {row['ap_std']:.3f} | {row['roc_auc_mean']:.3f} | {row['best_f1_mean']:.3f} |"
        )
    lines.append("")
    lines.append("## Same Models Scored Against Real Labels")
    lines.append("")
    lines.append("| feature set | spec | runs | prevalence | AP mean | AP std | ROC AUC mean | F1 mean |")
    lines.append("|---|---|---:|---:|---:|---:|---:|---:|")
    for _, row in real_test.iterrows():
        lines.append(
            f"| {row['feature_set']} | {row['spec']} | {int(row['runs'])} | {row['prevalence_mean']:.3f} | {row['ap_mean']:.3f} | {row['ap_std']:.3f} | {row['roc_auc_mean']:.3f} | {row['best_f1_mean']:.3f} |"
        )
    lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append("- A valid negative control should collapse near label prevalence / chance after label shuffling.")
    lines.append("- If shuffled-label AP stays high, the pipeline may have leakage or a metric bug.")
    lines.append("- Compare this control with the real 20-seed TCN AP means and the feature-ablation APs; those should be meaningfully higher than shuffled controls.")
    (run_dir / "negative_control_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    return summary


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    source, X, y_real, meta, feature_columns = load_sequence(args.sequence_run)
    run_dir = make_run_dir(args.run_name)
    groups = build_feature_groups(feature_columns)
    selected_groups = {name: groups[name] for name in args.groups if name in groups}
    if not selected_groups:
        raise SystemExit(f"No valid groups selected. Available: {sorted(groups)}")
    specs = [
        ("tcn_aug_bce", "tcn", True, "bce"),
        ("tcn_aug_focal", "tcn", True, "focal"),
    ]
    config = {
        "source_run": str(source),
        "shuffle_seeds": args.shuffle_seeds,
        "shuffle_mode": args.shuffle_mode,
        "groups": list(selected_groups),
        "specs": [spec[0] for spec in specs],
        "epochs": args.epochs,
        "patience": args.patience,
        "control": "parent-video temporal label sequences shuffled within each split",
    }
    write_json(run_dir / "config.json", config)
    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else args.device)

    all_metrics = []
    all_history = []
    all_manifests = []
    for shuffle_seed in args.shuffle_seeds:
        if args.shuffle_mode == "parent_sequence":
            y_control, manifest = make_parent_sequence_shuffle(y_real, meta, shuffle_seed)
        elif args.shuffle_mode == "window_exact":
            y_control, manifest = make_exact_window_shuffle(y_real, meta, shuffle_seed)
        else:
            raise SystemExit(f"Unknown shuffle mode: {args.shuffle_mode}")
        manifest["shuffle_seed"] = int(shuffle_seed)
        manifest["shuffle_mode"] = args.shuffle_mode
        all_manifests.append(manifest)
        meta_control = control_meta(meta, y_control)
        meta_control.to_csv(run_dir / "features" / f"control_sequence_index_seed{shuffle_seed}.csv", index=False)
        for group_name, idxs in selected_groups.items():
            X_group = X[:, :, idxs].astype(np.float32)
            for spec_name, kind, augment, loss in specs:
                model_name = f"seed{shuffle_seed}_{group_name}_{spec_name}_negative_control"
                print(f"training {model_name}")
                model_args = SimpleNamespace(
                    seed=int(shuffle_seed),
                    batch_size=args.batch_size,
                    lr=args.lr,
                    weight_decay=args.weight_decay,
                    epochs=args.epochs,
                    patience=args.patience,
                    loss=loss,
                    label_smoothing=args.label_smoothing,
                    focal_gamma=args.focal_gamma,
                )
                model, history, train_time_s, model_size_bytes = train_one_model(
                    model_name,
                    kind,
                    augment,
                    X_group,
                    y_control,
                    meta_control,
                    run_dir,
                    model_args,
                    device,
                )
                for row in history:
                    row["shuffle_seed"] = int(shuffle_seed)
                    row["shuffle_mode"] = args.shuffle_mode
                    row["feature_set"] = group_name
                    row["spec"] = spec_name
                all_history.extend(history)
                loader = DataLoader(SequenceDataset(X_group, y_control, np.arange(len(meta)), augment=False), batch_size=args.batch_size, shuffle=False, num_workers=0)
                probs, _ = predict_model(model, loader, device)
                pred = meta.copy()
                pred["shuffle_seed"] = int(shuffle_seed)
                pred["shuffle_mode"] = args.shuffle_mode
                pred["feature_set"] = group_name
                pred["spec"] = spec_name
                for h_idx, horizon in enumerate(HORIZONS):
                    pred[f"prob_{horizon:.1f}s"] = probs[:, h_idx]
                    pred[f"control_danger_within_{horizon:.1f}s"] = y_control[:, h_idx].astype(int)
                pred.to_csv(run_dir / "features" / f"predictions_{model_name}.csv", index=False)
                all_metrics.extend(
                    evaluate_probs(
                        model_name,
                        probs,
                        {"control_shuffled": y_control, "real_original": y_real},
                        meta,
                        group_name,
                        spec_name,
                        shuffle_seed,
                    )
                )
                for row in all_metrics[-(2 * 3 * len(HORIZONS)) :]:
                    row["shuffle_mode"] = args.shuffle_mode
                pd.DataFrame(all_metrics).to_csv(run_dir / "metrics" / "negative_control_metrics.csv", index=False)
                pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "negative_control_training_history.csv", index=False)

    if all_manifests:
        manifest_name = "parent_label_shuffle_manifest.csv" if args.shuffle_mode == "parent_sequence" else "window_label_shuffle_manifest.csv"
        pd.concat(all_manifests, ignore_index=True).to_csv(run_dir / "features" / manifest_name, index=False)
    metrics = pd.DataFrame(all_metrics)
    metrics.to_csv(run_dir / "metrics" / "negative_control_metrics.csv", index=False)
    pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "negative_control_training_history.csv", index=False)
    summarize(metrics, run_dir)
    append_report(
        run_dir,
        "Negative Control Completion",
        f"- Source run: `{source}`\n- Shuffle mode: `{args.shuffle_mode}`\n- Shuffle seeds: `{args.shuffle_seeds}`\n- Groups: `{list(selected_groups)}`\n- Summary: `{run_dir / 'negative_control_summary.md'}`",
    )
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Parent-video label-shuffle negative control for sequence danger models.")
    parser.add_argument("--sequence-run", default="runs/exp_008_sequence_len60_catalogue")
    parser.add_argument("--run-name", default="exp_024_sequence_negative_control")
    parser.add_argument("--shuffle-mode", choices=["parent_sequence", "window_exact"], default="parent_sequence")
    parser.add_argument("--shuffle-seeds", nargs="+", type=int, default=[701, 702, 703])
    parser.add_argument("--groups", nargs="+", default=["all_features", "no_zone_geometry"])
    parser.add_argument("--epochs", type=int, default=12)
    parser.add_argument("--patience", type=int, default=3)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--label-smoothing", type=float, default=0.05)
    parser.add_argument("--focal-gamma", type=float, default=2.0)
    parser.add_argument("--device", default="auto")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_024_sequence_negative_control_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["sequence_negative_control_experiments.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
